# Real Estate Investment Advisor
## Predicting Property Profitability & Future Value

**Project workflow:** data loading → cleaning → feature engineering → EDA → investment classification → five-year price forecasting → evaluation → model export → optional MLflow tracking.

> **Important data note:** The project brief names `india_housing_prices.csv`, but that CSV was not included with the two uploaded project documents. This notebook looks for that file. If it is not present, it creates a clearly labelled synthetic demonstration dataset so every cell can run. Replace the demo data with the actual CSV before reporting real-world results.

> **Important modelling note:** The brief does not provide historical five-year-ahead sale prices or a verified investment-return label. Therefore, this notebook creates transparent *proxy targets*: (1) “Good Investment” is based on a city/property-type median price-per-square-foot rule; (2) five-year value is estimated using an assumed annual appreciation rate (default 8%). These models learn the chosen rules; they do **not** establish actual future market performance. For production-quality forecasting, obtain time-stamped historical transactions and observed future sale prices.

## 1. Install and import dependencies

Run the install command only if a package is missing in your environment.

In [ ]:
# If needed, uncomment and run:
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib mlflow

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
DATA_PATH = Path("india_housing_prices.csv")
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
print("Setup complete.")

## 2. Load the dataset

Expected columns are listed in the project brief: location, property type, BHK, size, price, year built, furnishing, floor, nearby facilities, security, amenities, facing, owner type and availability status. Column names are normalized to simplify processing.

In [ ]:
def make_demo_data(n=1200, seed=RANDOM_STATE):
    """Create illustrative data only; never use this as real market evidence."""
    rng = np.random.default_rng(seed)
    states = ["Maharashtra", "Karnataka", "Telangana", "Tamil Nadu", "Assam", "Delhi"]
    cities_by_state = {
        "Maharashtra": ["Mumbai", "Pune", "Nagpur"],
        "Karnataka": ["Bengaluru", "Mysuru"],
        "Telangana": ["Hyderabad", "Warangal"],
        "Tamil Nadu": ["Chennai", "Coimbatore"],
        "Assam": ["Guwahati", "Dibrugarh"],
        "Delhi": ["New Delhi"]
    }
    property_types = ["Apartment", "Villa", "House"]
    furnishing = ["Unfurnished", "Semi-Furnished", "Fully-Furnished"]
    security = ["Basic", "CCTV", "Gated Community"]
    amenities = ["Basic", "Gym", "Gym, Pool", "Gym, Pool, Clubhouse"]
    facing = ["North", "South", "East", "West"]
    owner = ["Individual", "Builder", "Agent"]
    availability = ["Available", "Under Construction", "Sold"]

    rows = []
    for i in range(n):
        state = rng.choice(states)
        city = rng.choice(cities_by_state[state])
        ptype = rng.choice(property_types, p=[0.68, 0.16, 0.16])
        bhk = int(rng.choice([1, 2, 3, 4, 5], p=[.08, .36, .34, .17, .05]))
        size = max(350, int(rng.normal(450 + bhk * 430, 150)))
        year = int(rng.integers(1995, 2026))
        city_factor = {"Mumbai": 2.2, "Bengaluru": 1.7, "New Delhi": 2.0,
                       "Pune": 1.35, "Hyderabad": 1.25, "Chennai": 1.3,
                       "Guwahati": .75}.get(city, 1.0)
        type_factor = {"Apartment": 1.0, "Villa": 1.35, "House": 1.12}[ptype]
        price_per_sqft = max(1800, rng.normal(4200 * city_factor * type_factor, 700))
        price_lakhs = size * price_per_sqft / 100000
        locality = f"{city}_Locality_{int(rng.integers(1, 7))}"
        rows.append({
            "ID": i + 1, "State": state, "City": city, "Locality": locality,
            "Property_Type": ptype, "BHK": bhk, "Size_in_SqFt": size,
            "Price_in_Lakhs": round(price_lakhs, 2),
            "Price_per_SqFt": round(price_per_sqft, 2),
            "Year_Built": year, "Furnished_Status": rng.choice(furnishing),
            "Floor_No": int(rng.integers(0, 15)), "Total_Floors": int(rng.integers(1, 20)),
            "Nearby_Schools": int(rng.integers(0, 8)),
            "Nearby_Hospitals": int(rng.integers(0, 6)),
            "Public_Transport_Accessibility": rng.choice(["Low", "Medium", "High"]),
            "Parking_Space": int(rng.integers(0, 3)),
            "Security": rng.choice(security), "Amenities": rng.choice(amenities),
            "Facing": rng.choice(facing), "Owner_Type": rng.choice(owner),
            "Availability_Status": rng.choice(availability)
        })
    return pd.DataFrame(rows)

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE = "User-provided india_housing_prices.csv"
else:
    df = make_demo_data()
    DATA_SOURCE = "Synthetic demonstration data (replace with actual CSV)"

print("Data source:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())

## 3. Standardize columns and inspect data

The dataset description in the project document is the expected schema. The code tolerates optional columns, but `Price_in_Lakhs`, `Size_in_SqFt`, and at least one location field are needed for the two target definitions.

In [ ]:
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]

# Recalculate price per square foot when missing or invalid.
for col in ["Price_in_Lakhs", "Size_in_SqFt", "Year_Built"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "Price_per_SqFt" not in df.columns:
    df["Price_per_SqFt"] = np.nan
else:
    df["Price_per_SqFt"] = pd.to_numeric(df["Price_per_SqFt"], errors="coerce")

valid_ratio = (
    df["Price_in_Lakhs"].notna() & df["Size_in_SqFt"].notna()
    & (df["Size_in_SqFt"] > 0)
)
df.loc[df["Price_per_SqFt"].isna() & valid_ratio, "Price_per_SqFt"] = (
    df.loc[valid_ratio, "Price_in_Lakhs"] * 100000
    / df.loc[valid_ratio, "Size_in_SqFt"]
)

required = ["Price_in_Lakhs", "Size_in_SqFt"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df = df.drop_duplicates().copy()
df = df[(df["Price_in_Lakhs"] > 0) & (df["Size_in_SqFt"] > 0)].copy()
df = df.replace([np.inf, -np.inf], np.nan)

print("Cleaned shape:", df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values").head(20))
display(df.describe(include="all").T.head(25))

## 4. Feature engineering

Create age and price-per-square-foot features where possible. These transformations are deliberately simple and interpretable.

In [ ]:
CURRENT_YEAR = pd.Timestamp.today().year

if "Year_Built" in df.columns:
    df["Age_of_Property"] = (CURRENT_YEAR - df["Year_Built"]).clip(lower=0)
elif "Age_of_Property" not in df.columns:
    df["Age_of_Property"] = np.nan

if "Price_per_SqFt" not in df.columns:
    df["Price_per_SqFt"] = np.nan
df["Price_per_SqFt"] = df["Price_per_SqFt"].fillna(
    df["Price_in_Lakhs"] * 100000 / df["Size_in_SqFt"]
)

# Ensure common optional columns exist so downstream feature selection is stable.
optional_defaults = {
    "State": "Unknown", "City": "Unknown", "Locality": "Unknown",
    "Property_Type": "Unknown", "Furnished_Status": "Unknown",
    "Public_Transport_Accessibility": "Unknown", "Security": "Unknown",
    "Amenities": "Unknown", "Facing": "Unknown", "Owner_Type": "Unknown",
    "Availability_Status": "Unknown"
}
for col, default in optional_defaults.items():
    if col not in df.columns:
        df[col] = default

for col in ["BHK", "Floor_No", "Total_Floors", "Nearby_Schools",
            "Nearby_Hospitals", "Parking_Space"]:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Floor_Ratio"] = df["Floor_No"] / df["Total_Floors"].replace(0, np.nan)
df["Schools_per_1000_SqFt"] = df["Nearby_Schools"] / (df["Size_in_SqFt"] / 1000)

display(df.head())

## 5. Exploratory Data Analysis (EDA)

The charts below address the project brief's price/size, location, feature relationship, amenity and ownership questions. Interpret these as patterns in the supplied dataset, not causal effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["Price_in_Lakhs"], bins=35, kde=True, ax=axes[0])
axes[0].set_title("Distribution of property prices (lakhs)")
sns.histplot(df["Size_in_SqFt"], bins=35, kde=True, ax=axes[1])
axes[1].set_title("Distribution of property sizes (sq ft)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Size_in_SqFt", y="Price_in_Lakhs",
                hue="Property_Type", alpha=.6)
plt.title("Property size vs price")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
order = df.groupby("Property_Type")["Price_per_SqFt"].median().sort_values().index
sns.boxplot(data=df, x="Property_Type", y="Price_per_SqFt", order=order)
plt.title("Price per sq ft by property type")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

state_price = df.groupby("State", dropna=False)["Price_per_SqFt"].mean().sort_values(ascending=False)
display(state_price.to_frame("average_price_per_sqft"))
state_price.plot(kind="bar", figsize=(10, 4), title="Average price per sq ft by state")
plt.ylabel("Price per sq ft")
plt.tight_layout()
plt.show()

city_price = df.groupby("City", dropna=False)["Price_in_Lakhs"].mean().sort_values(ascending=False)
display(city_price.to_frame("average_price_lakhs"))
city_price.head(15).plot(kind="bar", figsize=(11, 4), title="Average property price by city")
plt.ylabel("Price (lakhs)")
plt.tight_layout()
plt.show()

numeric_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation heatmap of numeric features")
plt.tight_layout()
plt.show()

In [ ]:
# Additional EDA questions from the brief
eda_groupings = [
    ("Furnished_Status", "Price_in_Lakhs", "Price by furnishing status"),
    ("Facing", "Price_per_SqFt", "Price per sq ft by facing"),
    ("Owner_Type", "ID", "Property counts by owner type"),
    ("Availability_Status", "ID", "Property counts by availability status"),
    ("Public_Transport_Accessibility", "Price_per_SqFt", "Transport access vs price per sq ft"),
    ("Amenities", "Price_per_SqFt", "Amenities vs price per sq ft"),
    ("Parking_Space", "Price_in_Lakhs", "Parking spaces vs price")
]
for group_col, value_col, chart_title in eda_groupings:
    if group_col in df.columns:
        if value_col == "ID" and value_col not in df.columns:
            summary = df[group_col].value_counts()
        elif value_col == "ID":
            summary = df.groupby(group_col).size()
        else:
            summary = df.groupby(group_col)[value_col].median().sort_values(ascending=False)
        print(chart_title)
        display(summary.to_frame("count" if value_col == "ID" else "median"))
        summary.plot(kind="bar", figsize=(8, 3), title=chart_title)
        plt.tight_layout()
        plt.show()

for facility in ["Nearby_Schools", "Nearby_Hospitals"]:
    if facility in df.columns:
        plt.figure(figsize=(7, 4))
        sns.regplot(data=df, x=facility, y="Price_per_SqFt",
                    scatter_kws={"alpha": .35})
        plt.title(f"{facility} vs price per sq ft")
        plt.tight_layout()
        plt.show()

# Outlier counts using the IQR rule
outlier_summary = {}
for col in ["Price_per_SqFt", "Size_in_SqFt", "Price_in_Lakhs"]:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    outlier_summary[col] = int(((df[col] < q1 - 1.5 * iqr) |
                                (df[col] > q3 + 1.5 * iqr)).sum())
display(pd.Series(outlier_summary, name="IQR_outlier_count").to_frame())

## 6. Create transparent target variables

### Classification: `Good_Investment`
The project brief suggests comparing a property's price per sq ft with a median. This notebook uses the median for the same **city and property type**; when that group has too few records, it falls back to the city median and then the overall median. A property at or below the reference median is labelled 1 (good by this simplified price rule).

### Regression: `Future_Price_5Y_Lakhs`
The brief gives 8% as an example fixed annual rate. This notebook uses 8% by default:
\[
\text{Future Price} = \text{Current Price} \times (1 + r)^5
\]
This is an assumed scenario, not an observed target. Change the rate only when you have a defensible market assumption or historical data.

In [ ]:
ANNUAL_APPRECIATION_RATE = 0.08
FORECAST_YEARS = 5
MIN_GROUP_SIZE = 10

group_keys = ["City", "Property_Type"]
group_medians = df.groupby(group_keys)["Price_per_SqFt"].median()
group_counts = df.groupby(group_keys)["Price_per_SqFt"].size()
city_medians = df.groupby("City")["Price_per_SqFt"].median()
overall_median = df["Price_per_SqFt"].median()

def investment_reference(row):
    key = (row["City"], row["Property_Type"])
    if key in group_medians.index and group_counts.loc[key] >= MIN_GROUP_SIZE:
        return group_medians.loc[key]
    return city_medians.get(row["City"], overall_median)

df["Investment_Reference_Price_SqFt"] = df.apply(investment_reference, axis=1)
df["Good_Investment"] = (
    df["Price_per_SqFt"] <= df["Investment_Reference_Price_SqFt"]
).astype(int)

df["Future_Price_5Y_Lakhs"] = (
    df["Price_in_Lakhs"] * (1 + ANNUAL_APPRECIATION_RATE) ** FORECAST_YEARS
)

print("Good investment label distribution:")
display(df["Good_Investment"].value_counts(normalize=True).rename("share").to_frame())
print(f"Assumed annual appreciation: {ANNUAL_APPRECIATION_RATE:.1%}")
print(f"Five-year multiplier: {(1 + ANNUAL_APPRECIATION_RATE) ** FORECAST_YEARS:.4f}")

## 7. Prepare modelling features and train/test split

For classification, current price and price-per-square-foot are excluded from inputs because the label is directly defined using price per sq ft. This reduces direct target leakage, though location and property characteristics can still act as proxies. For regression, current price is intentionally included because it is the starting point for the five-year scenario formula.

In [ ]:
numeric_features = [
    "BHK", "Size_in_SqFt", "Age_of_Property", "Floor_No", "Total_Floors",
    "Nearby_Schools", "Nearby_Hospitals", "Parking_Space", "Floor_Ratio",
    "Schools_per_1000_SqFt"
]
categorical_features = [
    "State", "City", "Locality", "Property_Type", "Furnished_Status",
    "Public_Transport_Accessibility", "Security", "Amenities", "Facing",
    "Owner_Type", "Availability_Status"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

classification_features = numeric_features + categorical_features
regression_features = classification_features + ["Price_in_Lakhs"]

X_cls = df[classification_features].copy()
y_cls = df["Good_Investment"].copy()
X_reg = df[regression_features].copy()
y_reg = df["Future_Price_5Y_Lakhs"].copy()

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cls
)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

def make_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])

cls_preprocessor = make_preprocessor(numeric_features, categorical_features)
reg_preprocessor = make_preprocessor(
    numeric_features + ["Price_in_Lakhs"], categorical_features
)
print("Classification features:", len(classification_features))
print("Regression features:", len(regression_features))

## 8. Train classification models

Compare Logistic Regression and Random Forest is a common baseline approach; here Random Forest is used as the primary nonlinear model. Metrics include accuracy, precision, recall, F1 and ROC-AUC when both classes are present.

In [ ]:
from sklearn.linear_model import LogisticRegression

classification_models = {
    "Logistic Regression": Pipeline([
        ("preprocess", cls_preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", cls_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=350, min_samples_leaf=2,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

classification_results = []
for name, model in classification_models.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    row = {
        "Model": name,
        "Accuracy": accuracy_score(yc_test, pred),
        "Precision": precision_score(yc_test, pred, zero_division=0),
        "Recall": recall_score(yc_test, pred, zero_division=0),
        "F1": f1_score(yc_test, pred, zero_division=0)
    }
    if len(np.unique(yc_test)) == 2 and hasattr(model, "predict_proba"):
        row["ROC_AUC"] = roc_auc_score(yc_test, model.predict_proba(Xc_test)[:, 1])
    else:
        row["ROC_AUC"] = np.nan
    classification_results.append(row)

classification_results = pd.DataFrame(classification_results).sort_values("F1", ascending=False)
display(classification_results)
best_cls_name = classification_results.iloc[0]["Model"]
best_cls_model = classification_models[best_cls_name]
print("Selected classification model:", best_cls_name)

cls_predictions = best_cls_model.predict(Xc_test)
print(classification_report(yc_test, cls_predictions, zero_division=0))
cm = confusion_matrix(yc_test, cls_predictions)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion matrix — {best_cls_name}")
plt.tight_layout()
plt.show()

## 9. Train regression models

Because the target is generated from a fixed appreciation formula, the regression scores will mainly measure how well the model approximates that formula. Do not present these scores as evidence of real-world five-year forecasting accuracy.

In [ ]:
regression_models = {
    "Random Forest Regressor": Pipeline([
        ("preprocess", reg_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=350, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    "Extra Trees Regressor": Pipeline([
        ("preprocess", reg_preprocessor),
        ("model", ExtraTreesRegressor(
            n_estimators=350, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

regression_results = []
for name, model in regression_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(yr_test, pred),
        "RMSE": np.sqrt(mean_squared_error(yr_test, pred)),
        "R2": r2_score(yr_test, pred)
    })

regression_results = pd.DataFrame(regression_results).sort_values("RMSE")
display(regression_results)
best_reg_name = regression_results.iloc[0]["Model"]
best_reg_model = regression_models[best_reg_name]
print("Selected regression model:", best_reg_name)

reg_predictions = best_reg_model.predict(Xr_test)
plt.figure(figsize=(6, 6))
plt.scatter(yr_test, reg_predictions, alpha=.55)
lims = [min(yr_test.min(), reg_predictions.min()), max(yr_test.max(), reg_predictions.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual proxy future price (lakhs)")
plt.ylabel("Predicted future price (lakhs)")
plt.title(f"Actual vs predicted — {best_reg_name}")
plt.tight_layout()
plt.show()

## 10. Feature importance

Permutation importance measures how much the selected model's score changes when a feature is shuffled. It is a model diagnostic, not proof of causation.

In [ ]:
cls_perm = permutation_importance(
    best_cls_model, Xc_test, yc_test, n_repeats=5,
    random_state=RANDOM_STATE, scoring="f1"
)
cls_importance = pd.Series(
    cls_perm.importances_mean, index=classification_features
).sort_values(ascending=False).head(15)
display(cls_importance.to_frame("permutation_importance"))
cls_importance.sort_values().plot(kind="barh", figsize=(8, 5),
                                  title="Classification permutation importance")
plt.tight_layout()
plt.show()

reg_perm = permutation_importance(
    best_reg_model, Xr_test, yr_test, n_repeats=5,
    random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error"
)
reg_importance = pd.Series(
    reg_perm.importances_mean, index=regression_features
).sort_values(ascending=False).head(15)
display(reg_importance.to_frame("permutation_importance"))
reg_importance.sort_values().plot(kind="barh", figsize=(8, 5),
                                  title="Regression permutation importance")
plt.tight_layout()
plt.show()

## 11. Example prediction for a new property

Edit the values below. The classifier returns a model estimate of the rule-based label; the regression returns the estimated five-year scenario value.

In [ ]:
example = {
    "BHK": 3, "Size_in_SqFt": 1500, "Age_of_Property": 3,
    "Floor_No": 5, "Total_Floors": 12, "Nearby_Schools": 3,
    "Nearby_Hospitals": 2, "Parking_Space": 1, "Floor_Ratio": 5/12,
    "Schools_per_1000_SqFt": 2.0, "State": "Assam", "City": "Guwahati",
    "Locality": "Guwahati_Locality_1", "Property_Type": "Apartment",
    "Furnished_Status": "Semi-Furnished",
    "Public_Transport_Accessibility": "Medium", "Security": "CCTV",
    "Amenities": "Gym", "Facing": "East", "Owner_Type": "Builder",
    "Availability_Status": "Available", "Price_in_Lakhs": 65.0
}
example_df = pd.DataFrame([example])

cls_input = example_df[classification_features]
reg_input = example_df[regression_features]

investment_prediction = int(best_cls_model.predict(cls_input)[0])
future_price_prediction = float(best_reg_model.predict(reg_input)[0])

print("Good Investment prediction:", "YES" if investment_prediction == 1 else "NO")
if hasattr(best_cls_model, "predict_proba"):
    print("Model probability for Good Investment:",
          round(float(best_cls_model.predict_proba(cls_input)[0][1]), 3))
print(f"Estimated five-year price: ₹{future_price_prediction:.2f} lakhs")
print("Scenario assumption:", f"{ANNUAL_APPRECIATION_RATE:.1%} annual appreciation")

## 12. Save models and project metadata

The exported pipelines include preprocessing, so new data must use the same original feature names. Keep the metadata JSON with the model files.

In [ ]:
joblib.dump(best_cls_model, OUTPUT_DIR / "investment_classifier.joblib")
joblib.dump(best_reg_model, OUTPUT_DIR / "future_price_regressor.joblib")

metadata = {
    "data_source": DATA_SOURCE,
    "classification_model": best_cls_name,
    "regression_model": best_reg_name,
    "classification_features": classification_features,
    "regression_features": regression_features,
    "annual_appreciation_rate_assumption": ANNUAL_APPRECIATION_RATE,
    "forecast_years": FORECAST_YEARS,
    "classification_target_definition": (
        "1 when property price per sq ft <= city/property-type median; "
        "fallback to city median, then overall median"
    ),
    "regression_target_definition": (
        "current price * (1 + annual appreciation rate)^forecast years"
    ),
    "warning": "Targets are proxies, not observed investment outcomes or future sale prices."
}
with open(OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

classification_results.to_csv(OUTPUT_DIR / "classification_metrics.csv", index=False)
regression_results.to_csv(OUTPUT_DIR / "regression_metrics.csv", index=False)
print("Saved artifacts to:", OUTPUT_DIR.resolve())

## 13. Optional MLflow experiment tracking

MLflow is optional. If installed and enabled, this cell logs the selected models' parameters and evaluation metrics. For a formal model registry, configure a tracking server and follow your organization's model governance process.

In [ ]:
ENABLE_MLFLOW = False  # Change to True after installing MLflow if desired.

if ENABLE_MLFLOW:
    try:
        import mlflow
        import mlflow.sklearn

        mlflow.set_experiment("Real_Estate_Investment_Advisor")
        with mlflow.start_run(run_name="baseline_models"):
            mlflow.log_param("data_source", DATA_SOURCE)
            mlflow.log_param("classification_model", best_cls_name)
            mlflow.log_param("regression_model", best_reg_name)
            mlflow.log_param("annual_appreciation_rate", ANNUAL_APPRECIATION_RATE)
            for _, row in classification_results.iterrows():
                for metric in ["Accuracy", "Precision", "Recall", "F1"]:
                    mlflow.log_metric(f"classification_{row['Model']}_{metric}", float(row[metric]))
            for _, row in regression_results.iterrows():
                for metric in ["MAE", "RMSE", "R2"]:
                    mlflow.log_metric(f"regression_{row['Model']}_{metric}", float(row[metric]))
            mlflow.sklearn.log_model(best_cls_model, "investment_classifier")
            mlflow.sklearn.log_model(best_reg_model, "future_price_regressor")
        print("MLflow run logged.")
    except ImportError:
        print("MLflow is not installed. Run %pip install mlflow and retry.")
else:
    print("MLflow tracking skipped. Set ENABLE_MLFLOW = True to enable.")

## 14. Final project conclusions and limitations

- The notebook covers preprocessing, feature engineering, EDA, classification, regression, evaluation, feature importance and artifact export.
- The investment label is a transparent relative-price heuristic, not a guaranteed return indicator.
- The five-year price target is calculated using an assumed appreciation rate. Model performance on this target is not a genuine forecast validation.
- A reliable forecasting system needs historical property transactions with dates and actual resale values after the forecast horizon. A reliable investment label needs a defensible return definition that includes costs, rental income, taxes, liquidity and risk.
- Check for duplicate listings, inconsistent units, location leakage and time leakage before deploying.
- Do not use this output as financial or legal advice. Verify title, approvals, RERA registration, encumbrances, taxes and local market conditions independently.